# Behavioral Cloning with CarRacing-v3

Behavioral cloning (BC) is the simplest form of imitation learning: collect expert demonstrations, then train a policy via supervised learning to map observations to actions. It is the starting point for understanding why imitation learning works — and why it fails.

In this notebook you will:

1. Train an expert policy using PPO
2. Collect expert driving demonstrations
3. Train a BC policy via supervised learning on the expert's data
4. Observe **distribution shift** — the core failure mode of BC
5. Fix it with **DAgger** (Dataset Aggregation)

**Environment:** [CarRacing-v3](https://gymnasium.farama.org/environments/box2d/car_racing/) — a continuous-control driving task where the agent observes a 96×96 RGB top-down view of a procedurally generated race track and outputs steering, throttle, and brake.

## Setup

In [ ]:
!pip install -q "gymnasium[box2d]" stable-baselines3 swig

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Step 1: Create the environment

In [ ]:
def make_env():
    return gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")

venv = DummyVecEnv([make_env])
print(f"Observation space: {venv.observation_space}")
print(f"Action space: {venv.action_space}")

## Step 2: Train an expert policy

We train an expert using PPO. In a real setting you might use a pre-trained checkpoint or human teleoperation. Here PPO acts as our "expert driver."

**Note:** Training for 200k timesteps takes ~5-10 minutes on CPU.

In [ ]:
expert = PPO(
    "CnnPolicy",
    venv,
    verbose=1,
    seed=SEED,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    learning_rate=3e-4,
)

expert.learn(total_timesteps=200_000)
print("Expert training complete")

In [ ]:
expert_reward, expert_std = evaluate_policy(expert, venv, n_eval_episodes=10)
print(f"Expert mean reward: {expert_reward:.1f} +/- {expert_std:.1f}")

## Step 3: Collect expert demonstrations

Roll out the expert to collect (observation, action) pairs — this is our training data for behavioral cloning.

In [ ]:
def collect_demonstrations(policy, venv, n_episodes=50):
    """Collect (obs, action) pairs from a policy."""
    all_obs, all_actions = [], []
    for ep in range(n_episodes):
        obs = venv.reset()
        done = False
        while not done:
            action, _ = policy.predict(obs, deterministic=True)
            all_obs.append(obs[0].copy())
            all_actions.append(action[0].copy())
            obs, reward, done, info = venv.step(action)
    return np.array(all_obs), np.array(all_actions)

expert_obs, expert_actions = collect_demonstrations(expert, venv, n_episodes=50)
print(f"Collected {len(expert_obs)} transitions from 50 episodes")
print(f"Observation shape: {expert_obs.shape}")
print(f"Action shape: {expert_actions.shape}")

# Show sample observations
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    idx = i * (len(expert_obs) // 4)
    ax.imshow(expert_obs[idx])
    ax.set_title(f"Frame {idx}")
    ax.axis("off")
fig.suptitle("Sample expert observations")
plt.tight_layout()
plt.show()

## Step 4: Train a behavioral cloning policy

BC is supervised learning: a CNN maps observations to actions, trained with MSE loss against the expert's recorded actions.

In [ ]:
class BCPolicy(nn.Module):
    """CNN policy that maps 96x96 RGB observations to 3 continuous actions."""

    def __init__(self):
        super().__init__()
        # Input: (batch, 96, 96, 3) -> permute to (batch, 3, 96, 96)
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        # Calculate flattened size
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 96, 96)
            n_flat = self.features(dummy).shape[1]

        self.head = nn.Sequential(
            nn.Linear(n_flat, 256),
            nn.ReLU(),
            nn.Linear(256, 3),  # steering, throttle, brake
            nn.Tanh(),  # actions in [-1, 1]
        )

    def forward(self, x):
        # x: (batch, H, W, C) uint8 -> (batch, C, H, W) float32
        if x.dim() == 3:
            x = x.unsqueeze(0)
        x = x.permute(0, 3, 1, 2).float() / 255.0
        return self.head(self.features(x))

bc_policy = BCPolicy().to(device)
print(f"BC policy parameters: {sum(p.numel() for p in bc_policy.parameters()):,}")

In [ ]:
# Prepare data
obs_tensor = torch.tensor(expert_obs, dtype=torch.uint8)
act_tensor = torch.tensor(expert_actions, dtype=torch.float32)

dataset = TensorDataset(obs_tensor, act_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Train
optimizer = optim.Adam(bc_policy.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(20):
    epoch_loss = 0
    for obs_batch, act_batch in dataloader:
        obs_batch = obs_batch.to(device)
        act_batch = act_batch.to(device)

        pred_actions = bc_policy(obs_batch)
        loss = loss_fn(pred_actions, act_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/20  loss={avg_loss:.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("BC Training Loss")
plt.tight_layout()
plt.show()

## Step 5: Evaluate and observe distribution shift

Deploy the BC policy and compare against the expert.

**What you will observe:** the BC policy performs noticeably worse than the expert. On straight sections it may track the road, but on sharp turns it drifts off. Once off-track, it enters states the expert never demonstrated — predictions become unreliable, and the car spirals further off course.

This is **compounding error** from distribution shift: at training time, the policy only saw states along the expert's trajectory. At test time, any small deviation puts the agent in unfamiliar territory.

In [ ]:
def evaluate_bc(policy, venv, n_episodes=20):
    """Evaluate a PyTorch BC policy in the vectorized env."""
    rewards = []
    for _ in range(n_episodes):
        obs = venv.reset()
        total_reward = 0
        done = False
        while not done:
            with torch.no_grad():
                obs_t = torch.tensor(obs[0], dtype=torch.uint8).to(device)
                action = policy(obs_t).cpu().numpy().flatten()
            obs, reward, done, info = venv.step([action])
            total_reward += reward[0]
        rewards.append(total_reward)
    return rewards

def evaluate_sb3(policy, venv, n_episodes=20):
    """Evaluate an SB3 policy."""
    rewards = []
    for _ in range(n_episodes):
        obs = venv.reset()
        total_reward = 0
        done = False
        while not done:
            action, _ = policy.predict(obs, deterministic=True)
            obs, reward, done, info = venv.step(action)
            total_reward += reward[0]
        rewards.append(total_reward)
    return rewards

expert_rewards = evaluate_sb3(expert, venv, n_episodes=20)
bc_rewards = evaluate_bc(bc_policy, venv, n_episodes=20)

print(f"Expert: {np.mean(expert_rewards):.1f} +/- {np.std(expert_rewards):.1f}")
print(f"BC:     {np.mean(bc_rewards):.1f} +/- {np.std(bc_rewards):.1f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([expert_rewards, bc_rewards], labels=["Expert (PPO)", "Behavioral Cloning"])
ax.set_ylabel("Episode Reward")
ax.set_title("Expert vs BC: Distribution Shift Degrades Performance")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Step 6: Fix it with DAgger

[DAgger](https://arxiv.org/abs/1011.0686) (Dataset Aggregation) addresses distribution shift by iteratively collecting new data from the *learner's* trajectory, labeled by the *expert*.

The algorithm:
1. Train an initial BC policy on expert demonstrations
2. Roll out the **learner's** policy in the environment
3. Ask the **expert** to label the states the learner visited (what would you have done here?)
4. Add this new data to the training set
5. Retrain and repeat

In [ ]:
def dagger_round(bc_policy, expert, venv, n_episodes=10):
    """Run one round of DAgger: roll out the learner, label with expert."""
    new_obs, new_actions = [], []
    for _ in range(n_episodes):
        obs = venv.reset()
        done = False
        while not done:
            # Learner drives
            with torch.no_grad():
                obs_t = torch.tensor(obs[0], dtype=torch.uint8).to(device)
                learner_action = bc_policy(obs_t).cpu().numpy().flatten()

            # Expert labels what IT would have done in this state
            expert_action, _ = expert.predict(obs, deterministic=True)

            new_obs.append(obs[0].copy())
            new_actions.append(expert_action[0].copy())

            # Learner's action determines next state
            obs, reward, done, info = venv.step([learner_action])

    return np.array(new_obs), np.array(new_actions)


def train_bc_on_data(bc_policy, all_obs, all_actions, n_epochs=5, lr=1e-4):
    """Retrain BC policy on accumulated data."""
    dataset = TensorDataset(
        torch.tensor(all_obs, dtype=torch.uint8),
        torch.tensor(all_actions, dtype=torch.float32),
    )
    loader = DataLoader(dataset, batch_size=64, shuffle=True)
    optimizer = optim.Adam(bc_policy.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(n_epochs):
        for obs_batch, act_batch in loader:
            pred = bc_policy(obs_batch.to(device))
            loss = loss_fn(pred, act_batch.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


# Start DAgger with the original expert data
all_obs = expert_obs.copy()
all_actions = expert_actions.copy()

# Reset BC policy for fair comparison
dagger_policy = BCPolicy().to(device)

# Initial BC training
train_bc_on_data(dagger_policy, all_obs, all_actions, n_epochs=20, lr=3e-4)

dagger_progress = []
N_ROUNDS = 5

for r in range(N_ROUNDS):
    # Collect new data from learner's trajectory, labeled by expert
    new_obs, new_actions = dagger_round(dagger_policy, expert, venv, n_episodes=10)

    # Aggregate
    all_obs = np.concatenate([all_obs, new_obs])
    all_actions = np.concatenate([all_actions, new_actions])

    # Retrain on full dataset
    train_bc_on_data(dagger_policy, all_obs, all_actions, n_epochs=5, lr=1e-4)

    # Evaluate
    rewards = evaluate_bc(dagger_policy, venv, n_episodes=10)
    mean_r = np.mean(rewards)
    dagger_progress.append(mean_r)
    print(f"DAgger round {r+1}/{N_ROUNDS}  data={len(all_obs):,}  reward={mean_r:.1f}")

print(f"\nFinal dataset size: {len(all_obs):,} transitions")

### Compare all three policies

In [ ]:
dagger_rewards = evaluate_bc(dagger_policy, venv, n_episodes=20)

print(f"Expert: {np.mean(expert_rewards):.1f} +/- {np.std(expert_rewards):.1f}")
print(f"BC:     {np.mean(bc_rewards):.1f} +/- {np.std(bc_rewards):.1f}")
print(f"DAgger: {np.mean(dagger_rewards):.1f} +/- {np.std(dagger_rewards):.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Box plot comparison
ax1.boxplot(
    [expert_rewards, bc_rewards, dagger_rewards],
    labels=["Expert (PPO)", "Behavioral Cloning", "DAgger"],
)
ax1.set_ylabel("Episode Reward")
ax1.set_title("Expert vs BC vs DAgger")
ax1.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

# DAgger learning curve
ax2.plot(range(1, N_ROUNDS + 1), dagger_progress, "o-", color="green")
ax2.axhline(y=np.mean(bc_rewards), color="orange", linestyle="--", label="BC baseline")
ax2.axhline(y=np.mean(expert_rewards), color="blue", linestyle="--", label="Expert")
ax2.set_xlabel("DAgger Round")
ax2.set_ylabel("Mean Reward")
ax2.set_title("DAgger Improvement Over Rounds")
ax2.legend()

plt.tight_layout()
plt.show()

## Summary

| Policy | Training method | Distribution shift? |
|---|---|---|
| **Expert (PPO)** | RL with environment reward | N/A — defines the target distribution |
| **Behavioral Cloning** | Supervised regression on expert data | Yes — compounds errors on unseen states |
| **DAgger** | Iterative BC with learner-visited states labeled by expert | Mitigated — training distribution converges to test distribution |

The distribution shift problem you observed here — and DAgger's fix of relabeling learner-visited states — reappears throughout robot learning. Later in the course, you will revisit these ideas in the context of world models and VLA architectures, where the same fundamental challenge is addressed at larger scale.

## Further reading

- Bagnell (2015). [An Invitation to Imitation](https://www.ri.cmu.edu/pub_files/2015/3/InvitationToImitation_3_1415.pdf) — accessible introduction to the theory
- Ross et al. (2011). [A Reduction of Imitation Learning and Structured Prediction to No-Regret Online Learning](https://arxiv.org/abs/1011.0686) — the DAgger paper
- Bojarski et al. (2016). [End to End Learning for Self-Driving Cars](https://arxiv.org/abs/1604.07316) — NVIDIA's original end-to-end driving work
- The [imitation library documentation](https://imitation.readthedocs.io/en/latest/tutorials/1_train_bc.html) provides complete tutorials for BC, DAgger, GAIL, and AIRL